# Last Modified: 2026-04-10 11:54:10

---

# LFP Battery SOH Preprocessing Pipeline (Batch Processing Version)

This notebook implements a physically-consistent preprocessing framework for the entire dataset:
1. **Phase 1**: Global Physical Cleaning (Unit conversion and noise smoothing for ALL cells)
2. **Phase 2**: Scenario-based Slicing & Dual Labeling (SOC & Mode)
3. **Phase 3**: 40D HI Extraction (Using physical units: V, A, s, °C)
4. **Phase 4**: Batch Feature Scaling (Z-score or Min-Max)
5. **Phase 5**: Global Tensor Generation & Saving

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from tqdm.notebook import tqdm
import os
import sys
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import skew, kurtosis
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA
import pdb

# Add project root to sys.path to allow importing from src
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import load_hust, load_mit

## Phase 0: Data Loading

Load all raw data into memory.

In [5]:
DATASET_TYPE = "hust"
all_cells = load_hust(PROJECT_ROOT / "HUST_data" / "data") if DATASET_TYPE == "hust" else load_mit(PROJECT_ROOT / "MIT_data")
print(f"Loaded {len(all_cells)} cells for {DATASET_TYPE.upper()} dataset.")

HUST Cells: 100%|██████████| 77/77 [00:54<00:00,  1.41it/s]

Loaded 77 cells for HUST dataset.


## Phase 1: Global Physical Cleaning & Unit Conversion

Apply cleaning to **ALL** cells and **ALL** cycles immediately to ensure physical consistency.

In [ ]:
def clean_physical_data(df):
    clean_df = df.copy()
    v_col = [c for c in df.columns if "Voltage" in c][0]
    i_col = [c for c in df.columns if "Current" in c][0]
    
    # 1. Unit conversion (mA -> A for HUST)
    if "Current (mA)" in clean_df.columns: clean_df["Current (A)"] = clean_df["Current (mA)"] / 1000.0
    
    # # 2. Advanced interpolation (Numeric only to avoid TypeError)
    num_cols = clean_df.select_dtypes(include=[np.number]).columns
    clean_df[num_cols] = clean_df[num_cols].interpolate(method='linear', limit_direction='both')
    clean_df = clean_df.ffill().bfill()
    
    # 3. Noise Smoothing
    clean_df[v_col] = savgol_filter(clean_df[v_col], window_length=11, polyorder=3)

    (df[v_col]-clean_df[v_col]).describe().to_csv(PROJECT_ROOT / "outputs" / "cleaning_report.csv")
    pdb.set_trace()

    return clean_df

# Global Execution for All Cells
for cid, cell in tqdm(all_cells.items(), desc="Global Physical Cleaning"):
    for cyc in cell.data.keys():
        cell.data[cyc] = clean_physical_data(cell.data[cyc])

print("Global Physical Cleaning and Unit Conversion complete for the entire dataset.")

Global Physical Cleaning:   0%|          | 0/77 [00:00<?, ?it/s]

> c:\users\ksshin\appdata\local\temp\ipykernel_46016\843064211.py(20)clean_physical_data()

> c:\users\ksshin\appdata\local\temp\ipykernel_46016\843064211.py(20)clean_physical_data()



## Phase 2: Scenario-based Slicing & Dual Labeling (SOC & Mode)

Define slicing logic with Mode Labels: Discharge (0), Charge (1).

In [ ]:
def phase2_slice_data(df, start_p=0.0, end_p=1.0, mode="C"): 
    col_i = "Current (A)" if "Current (A)" in df.columns else [c for c in df.columns if "Current" in c][0]
    mode_df = df[df[col_i] > 0.01] if mode == "C" else df[df[col_i] < -0.01]
    if mode_df.empty: return mode_df, 0, (1 if mode == "C" else 0)
    
    n = len(mode_df)
    s_idx, e_idx = int(n * start_p), int(n * end_p)
    sliced = mode_df.iloc[s_idx:e_idx]
    
    # SOC Labeling: H(-2), M(-1), L(0)
    avg_pos = (start_p + end_p) / 2
    soc_label = -2 if avg_pos <= 0.3 else (-1 if avg_pos <= 0.7 else 0)
    
    # Mode Labeling: Discharge(0), Charge(1)
    mode_label = 1 if mode == "C" else 0
    
    return sliced, soc_label, mode_label

## Phase 3: 40D Adaptive HI Pool Extraction (Dual Mode)

Extract features from any sliced segment.

In [ ]:
def extract_40d_hi(df, mode="C", history_df=None): 
    v_col = [c for c in df.columns if "Voltage" in c][0]
    i_col = "Current (A)" if "Current (A)" in df.columns else [c for c in df.columns if "Current" in c][0]
    t_col = [c for c in df.columns if "Time" in c][0]
    temp_col = [c for c in df.columns if "Temperature" in c]
    v, i, t = df[v_col].values, df[i_col].values, df[t_col].values
    temp = df[temp_col[0]].values if temp_col else np.full_like(v, 25.0) + np.random.normal(0, 0.05, len(v))
    dt = np.diff(t, prepend=t[0])
    his = {}
    
    # Category 1: Common (14)
    his["c1_mean_v"], his["c1_var_v"] = np.mean(v), np.var(v)
    his["c1_skew_v"], his["c1_kurt_v"] = skew(v), kurtosis(v)
    his["c1_mean_t"], his["c1_delta_t"] = np.mean(temp), np.max(temp)-np.min(temp)
    his["c1_dtdt"] = np.mean(np.diff(temp)/np.where(dt[1:]>0, dt[1:], 1e-6))
    his["c1_relax_v"] = v[-1] - v[0]
    his["c1_relax_tc"] = np.log(np.abs(v[-1]-v[-2])+1e-6)
    his["c1_var_i"] = np.var(i)
    his["c1_ent_v"] = -np.sum(np.histogram(v, bins=10, density=True)[0] * np.log(np.histogram(v, bins=10, density=True)[0] + 1e-6))
    his["c1_ent_t"] = -np.sum(np.histogram(temp, bins=10, density=True)[0] * np.log(np.histogram(temp, bins=10, density=True)[0] + 1e-6))
    his["c1_de_dq"] = np.sum(v*i*dt) / (np.sum(i*dt)+1e-6)
    his["c1_ma"] = history_df["c1_mean_v"].iloc[-5:].mean() if history_df is not None and not history_df.empty else his["c1_mean_v"]

    for j in range(1, 14): his[f"c2_dis_{j}"] = his[f"c3_cha_{j}"] = 0.0
    if mode == "D":
        c_rate = np.abs(i)/1.1
        his["c2_dis_1"] = np.sum(np.diff(np.round(c_rate, 1)) != 0)
        his["c2_dis_2"] = np.mean(np.abs(np.diff(v)/(np.diff(i)+1e-6)))
        his["c2_dis_3"] = np.trapezoid(v-np.min(v), t)/(t[-1]-t[0])
        his["c2_dis_4"] = t[np.argmin(v>=3.0)] if np.any(v<3.0) else t[-1]
        his["c2_dis_5"] = np.sum((v<=3.2)&(v>=3.1)) * np.mean(dt)
        dvdt = np.diff(v)/np.where(dt[1:]>0, dt[1:], 1e-6)
        his["c2_dis_6"], his["c2_dis_7"] = np.mean(dvdt), np.var(dvdt)
        his["c2_dis_12"] = np.min(np.diff(v)/np.where(np.diff(np.cumsum(np.abs(i)*dt)/3600.0)>0, np.diff(np.cumsum(np.abs(i)*dt)/3600.0), 1e-6))
    elif mode == "C":
        is_cv, q_cum = np.abs(np.diff(v))<0.001, np.cumsum(np.abs(i)*dt)/3600.0
        if np.any(is_cv):
            cv_s = np.where(is_cv)[0][0]
            his["c3_cha_1"], his["c3_cha_2"], his["c3_cha_3"] = q_cum[cv_s]/q_cum[-1], t[cv_s]-t[0], t[-1]-t[cv_s]
        his["c3_cha_8"] = q_cum[(v>=3.35)&(v<=3.45)][-1] - q_cum[(v>=3.35)&(v<=3.45)][0] if np.any((v>=3.35)&(v<=3.45)) else 0.0
        dqdv = np.diff(q_cum)/np.where(np.diff(v)>0.0001, np.diff(v), 1e-6)
        his["c3_cha_9"], his["c3_cha_10"] = np.max(dqdv), np.min(dqdv)
    return pd.Series(his)

## Phase 4: Batch Processing & Feature Generation

Iterate through all cleaned cells and cycles to extract features for the entire dataset.

In [ ]:
SCALING_METHOD = "zscore"
import pdb

def phase5_save_dataset(cells, dataset_type):
    out_path = PROJECT_ROOT / "outputs" / "processed" / f"{dataset_type}_all_tensors_batch.pkl"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    final_data = []
    
    scen_bounds = {"H": (0.0, 0.3), "M": (0.3, 0.7), "L": (0.7, 1.0), "Random": (0.0, 1.0)}
    lengths = [0.1, 0.2, 0.3]
    step = 0.02
    
    for cid, cell in tqdm(cells.items(), desc=f"Batch Processing {dataset_type.upper()}"):
        hist, hi_cyc_list = pd.DataFrame(), []

        pdb.set_trace()

        for cyc in sorted(list(cell.data.keys())):
            # Data is ALREADY cleaned in Phase 1
            c_df = cell.data[cyc]
            pdb.set_trace()
            
            # History Reference (Full Charge)
            ref_sliced, _, _ = phase2_slice_data(c_df, 0.0, 1.0, "C")
            if not ref_sliced.empty: ref_hi = extract_40d_hi(ref_sliced, "C", hist)
            pdb.set_trace()

            # Sliding Window Segments (Charge & Discharge)
            for mode in ["C", "D"]:
                for scen_name, (s_bound, e_bound) in scen_bounds.items():
                    for length in lengths:
                        max_start = e_bound - length
                        if max_start < s_bound: continue
                        for start_p in np.arange(s_bound, max_start + 0.0001, step):
                            end_p = start_p + length
                            sliced, soc_l, mode_l = phase2_slice_data(c_df, start_p, end_p, mode)
                            if sliced.empty or len(sliced) < 5: continue
                            
                            pdb.set_trace()
                            hi = extract_40d_hi(sliced, mode, hist)
                            hi_cyc_list.append({
                                "cyc": cyc, "raw": hi, 
                                "soc_label": soc_l, "mode_label": mode_l, 
                                "rul": cell.rul[cyc]
                            })
            
            if not ref_sliced.empty: hist = pd.concat([hist, ref_hi.to_frame().T], ignore_index=True)
        
        if not hi_cyc_list: continue
        
        # Local Scaling per Cell
        scaler = StandardScaler() if SCALING_METHOD == "zscore" else MinMaxScaler()
        scaled_feats = scaler.fit_transform(pd.DataFrame([x["raw"] for x in hi_cyc_list]))
        
        for i, item in enumerate(hi_cyc_list):
            final_data.append({
                "cell": cid, "cyc": item["cyc"], "x": scaled_feats[i], 
                "soc_label": item["soc_label"], "mode_label": item["mode_label"], 
                "y": item["rul"]
            })
            
    with open(out_path, "wb") as f: pickle.dump(final_data, f)
    print(f"Batch Success! Saved {len(final_data)} segments to {out_path}")

phase5_save_dataset(all_cells, DATASET_TYPE)